# 1 · `EngramMiddleware` — drop-in agent memory

`EngramMiddleware` gives a `create_agent` agent long-term memory with **no**
changes to the agent body:

- **before each model call** it searches Engram and injects relevant memories
  into the system prompt;
- **after the agent finishes** it hands the conversation to Engram, whose
  pipeline extracts and stores durable facts.

Requires an Engram key **and** an LLM key (Anthropic here).

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.
> For the agent cells you also need `langchain-anthropic` (Option A installs it; for Option B run `%pip install langchain-anthropic`).

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".." langchain-anthropic

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY


def seed_memory(text, user_id, **kwargs):
    """Add a memory and block until the async pipeline commits it."""
    run = client.memories.add(text, user_id=user_id, **kwargs)
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(query=query, user_id=user_id, **kwargs)
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(query=query, user_id=user_id, **kwargs)

### Deterministic recall demo
Engram's write pipeline is asynchronous, so for a repeatable demo we first
**seed** a fact and wait for it to commit. Then a brand-new agent thread
recalls it — proving cross-session memory.

In [ ]:
USER = 'demo-alice'
seed_memory('The user only drinks decaf coffee.', user_id=USER)

In [ ]:
from langchain.agents import create_agent

from langchain_engram import EngramMiddleware

agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware(user_id=USER)],
)

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Recommend a coffee for me.'}]}
)
print(result['messages'][-1].content)

The recommendation should respect the decaf preference, even though this
thread never mentioned it.

### Auto-write
Now let the middleware *learn* from a conversation. The write is
fire-and-forget, so we poll until Engram has committed the new fact.

In [ ]:
agent.invoke(
    {'messages': [{'role': 'user',
                   'content': 'By the way, I am vegetarian and allergic to peanuts.'}]}
)

found = wait_until_searchable('dietary restrictions', USER, 'vegetarian')
for m in found:
    print('-', m.content)

### Resolving `user_id` dynamically
Instead of binding a fixed user, read it from the agent's runtime context —
handy for multi-tenant apps.

In [ ]:
from typing import TypedDict


class Ctx(TypedDict):
    user_id: str


ctx_agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware()],  # no user_id -> read from context
    context_schema=Ctx,
)

out = ctx_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'What do I drink?'}]},
    context={'user_id': 'demo-alice'},
)
print(out['messages'][-1].content)